<a href="https://colab.research.google.com/github/Yusef-Hazem/Lang-Frameworks/blob/main/01FirstRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Python 3.12.13
# !pip install -U langchain==1.3.11
# !pip install requests==2.34.2
# !pip install -U langchain_community==0.4.2
# !pip install -U python-dotenv==1.2.2
# !pip install -U langchain-huggingface==1.2.2
# !pip install faiss-cpu==1.14.3
!pip install langchain-groq==1.1.3

In [ ]:
!pip show langchain-groq

In [ ]:
#create .env file
with open(".env", "w") as f:
    f.write("API_KEY=your_api_key\n")

In [ ]:
from dotenv import load_dotenv , find_dotenv
load_dotenv(find_dotenv())

True

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import bs4
import os

In [ ]:
# set enviroment variable of the groq APi key
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
loader = WebBaseLoader(web_path="https://lilianweng.github.io/posts/2023-06-23-agent/" ,
                       bs_kwargs={
                           "parse_only"  :bs4.SoupStrainer( class_ = ("post-single" , "post-content" , "post-title"))
                        })

In [ ]:
doc = loader.load()

In [ ]:
# chunking

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(doc)


In [ ]:
#embedding
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}

embed_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs)


In [ ]:
#vector DB
vector_db = FAISS.from_documents(documents = chunks , embedding = embed_model)
retriver = vector_db.as_retriever()

In [ ]:
llm= ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)

In [ ]:
def format_retrived_docs (docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant"),
    ("human", "\n check this : {context} \n\n Answer this {question} ")
])


In [ ]:
rag_chain = (
    {"context" : retriver| format_retrived_docs , "question" : RunnablePassthrough()} |
    prompt |
    llm|
    StrOutputParser()
)

In [ ]:
print(rag_chain.invoke("what is task decompostion ?"))

Task decomposition refers to the process of breaking down a large, complex task into smaller, more manageable subgoals or tasks. This can be done in various ways, such as:

1. Using a Large Language Model (LLM) with simple prompting, like asking for steps or subgoals.
2. Providing task-specific instructions, like writing a story outline.
3. With human input, where a person helps to break down the task into smaller parts.

The goal of task decomposition is to enable efficient handling of complex tasks by dividing them into smaller, more manageable pieces. This allows for better planning, reflection, and refinement, ultimately leading to improved results.

In the context of the AI assistant, task decomposition is used to parse user input into smaller tasks, which can be executed in a specific order, with dependencies between tasks. The "dep" field in the task definition denotes the id of the previous task that generates a new resource that the current task relies on.
